In [ ]:
RUN_TARGET = "colab"  # "colab" | "local"

## Setup Instructions

### Running on Google Colab
1. Set `RUN_TARGET = "colab"` in Cell 1
2. Runtime > Change runtime type > T4 GPU or A100
3. Run the pip-install cell (Cell 3) — this pins numpy and installs captum
4. Run the Drive-mount cell (Cell 4) — approve the Google Drive permission
   prompt now so you do not need to attend the runtime later.
   **Prerequisite:** notebook 03 must have already saved
   `baseline_frozen_esm2.pt` to Google Drive > My Drive > XAllergen2.0 > models/
5. Run the setup cell (Cell 5) to download data and copy the checkpoint
6. Run remaining cells normally
7. Results are automatically copied to
   Google Drive > My Drive > XAllergen2.0 > results/

### Running locally on macOS (M-series)
1. Set `RUN_TARGET = "local"` in Cell 1
2. Cells 3–5 are skipped automatically
3. MPS is used when available, otherwise CPU
4. Results are saved to the local `results/` directory

In [ ]:
if RUN_TARGET == "colab":
    import subprocess as _sp
    import sys as _sys

    # Colab (Python 3.12, torch 2.6+) ships with numpy 2.x, which torch requires.
    # Do NOT pin or downgrade numpy — torch is compiled against numpy 2.x and will
    # break with a dtype ABI error if numpy is downgraded to <2.0.
    # captum and transformers both support numpy 2.x; just install them directly.
    _sp.run(
        [_sys.executable, "-m", "pip", "install", "-q",
         "captum", "transformers>=4.30.0", "statsmodels"],
        check=True,
    )
    print("Colab environment ready.")
else:
    print("Local environment detected. Skipping Colab setup.")

In [ ]:
if RUN_TARGET == "colab":
    from google.colab import drive as _drive
    from pathlib import Path
    _drive.mount("/content/drive", force_remount=False)

    DRIVE_ROOT    = Path("/content/drive/MyDrive/XAllergen2.0")
    DRIVE_MODELS  = DRIVE_ROOT / "models"
    DRIVE_RESULTS = DRIVE_ROOT / "results"

    # Verify that notebook 03 has already saved the checkpoint to Drive.
    _ckpt_on_drive = DRIVE_MODELS / "baseline_frozen_esm2.pt"
    if not _ckpt_on_drive.exists():
        raise FileNotFoundError(
            f"Checkpoint not found on Drive: {_ckpt_on_drive}\n"
            "Run 03_baseline_model_colab.ipynb first to train and save the model."
        )
    print(f"Google Drive mounted. Checkpoint verified at: {_ckpt_on_drive}")
    print(f"Probing results will be written to: {DRIVE_RESULTS}")
else:
    print("Local run: skipping Google Drive mount.")

In [ ]:
if RUN_TARGET == "colab":
    import shutil as _shutil
    import urllib.request as _urlreq
    from pathlib import Path

    RUNTIME_ROOT = Path("/content/XAllergen2.0")
    _DATA_DIR  = RUNTIME_ROOT / "data"
    _MODEL_DIR = RUNTIME_ROOT / "models"
    for _d in [_DATA_DIR, _MODEL_DIR, RUNTIME_ROOT / "results"]:
        _d.mkdir(parents=True, exist_ok=True)

    _RAW = "https://raw.githubusercontent.com/Jeffateth/XAllergen2.0/main"

    # Utility module (must be on sys.path before the imports cell runs)
    _urlreq.urlretrieve(
        f"{_RAW}/baseline_notebook_utils.py",
        RUNTIME_ROOT / "baseline_notebook_utils.py",
    )
    print("Downloaded: baseline_notebook_utils.py")

    # IEDB epitope data
    _urlreq.urlretrieve(
        f"{_RAW}/data/positives_splitB.csv",
        _DATA_DIR / "positives_splitB.csv",
    )
    print("Downloaded: positives_splitB.csv")

    # Trained checkpoint — copy from Drive (saved there by notebook 03)
    _shutil.copy2(
        DRIVE_MODELS / "baseline_frozen_esm2.pt",
        _MODEL_DIR / "baseline_frozen_esm2.pt",
    )
    print(f"Copied checkpoint from Drive to runtime: {_MODEL_DIR / 'baseline_frozen_esm2.pt'}")
    print("Runtime setup complete.")
else:
    print("Local run: skipping GitHub download and Drive copy.")

# 04 — Probing Metrics

Evaluates the mechanistic faithfulness of `FrozenESMAllergenClassifier` by comparing
residue-level attribution maps against experimentally validated IEDB epitope annotations
from `positives_splitB.csv`.

Supports both Google Colab (`RUN_TARGET = "colab"`, T4/A100 GPU) and local
execution on macOS (`RUN_TARGET = "local"`, MPS or CPU).


In [1]:
import sys
from pathlib import Path

if RUN_TARGET == "colab":
    # RUNTIME_ROOT was set in the setup cell; add it to sys.path so that
    # baseline_notebook_utils (downloaded there) can be imported.
    if str(RUNTIME_ROOT) not in sys.path:
        sys.path.insert(0, str(RUNTIME_ROOT))

from baseline_notebook_utils import (
    DROPOUT,
    ESM_MODEL_NAME,
    HF_MODEL_NAME,
    HIDDEN_DIM,
    IG_STEPS,
    MAX_SEQ_LEN,
    RANDOM_STATE,
    build_tokenizer,
    compute_attention_weights,
    compute_integrated_gradients,
    configure_matplotlib_cache,
    detect_device,
    find_project_root,
    load_baseline_checkpoint,
    mean_metric_dicts,
    print_runtime_context,
    seed_everything,
)

if RUN_TARGET == "colab":
    import matplotlib
    matplotlib.use("Agg")  # headless-safe backend on Colab
else:
    configure_matplotlib_cache(Path.cwd())

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import average_precision_score, roc_auc_score
from statsmodels.nonparametric.smoothers_lowess import lowess
from tqdm.auto import tqdm

N_RANDOM_DRAWS = 100

if RUN_TARGET == "colab":
    import torch
    PROJECT_ROOT = RUNTIME_ROOT
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {DEVICE}")
    if DEVICE == "cuda":
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
        print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    else:
        print("  WARNING: no GPU detected — IG attribution will be slow.")
else:
    PROJECT_ROOT = find_project_root(Path.cwd())
    DEVICE = detect_device()
    print_runtime_context(DEVICE, PROJECT_ROOT)

DATA_DIR    = PROJECT_ROOT / "data"
MODEL_DIR   = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

POSITIVES_CSV     = DATA_DIR   / "positives_splitB.csv"
CHECKPOINT_PATH   = MODEL_DIR  / "baseline_frozen_esm2.pt"
SUMMARY_CSV       = RESULTS_DIR / "probing_summary.csv"
VIOLINS_PNG       = RESULTS_DIR / "probing_violins.png"
AUROC_DENSITY_PNG = RESULTS_DIR / "probing_auroc_vs_density.png"
AUPRC_DENSITY_PNG = RESULTS_DIR / "probing_auprc_vs_density.png"

seed_everything(RANDOM_STATE)


/Users/jianzhouyao/Library/Mobile Documents/com~apple~CloudDocs/Universität/ETH/DL in Biology/XAllergen2.0/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RUN_TARGET: local
Device: mps
Project root: /Users/jianzhouyao/Library/Mobile Documents/com~apple~CloudDocs/Universität/ETH/DL in Biology/XAllergen2.0
GPU configuration:
  backend: Apple Metal Performance Shaders (MPS)
  built with MPS: True
  MPS available: True


In [2]:
missing = [path for path in [POSITIVES_CSV, CHECKPOINT_PATH] if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Required files not found:\n" + "\n".join(f"  {path}" for path in missing)
        + ("\nRun the setup cell first." if RUN_TARGET == "colab" else "")
    )
print("All required files verified.")


All required local files verified.


## Data — Load and Parse Epitope Labels

In [9]:
raw_df = pd.read_csv(POSITIVES_CSV)
raw_df["accession"] = raw_df["accession"].astype(str)
raw_df["sequence"]  = raw_df["sequence"].astype(str).str.strip().str.upper()


def parse_epitope_label(
    sequence: str, epitope_start: str, epitope_end: str
) -> np.ndarray:
    """Build a binary per-residue label vector from semicolon-separated
    interval boundaries (1-indexed, inclusive).

    epitope_start = "14;45"  and  epitope_end = "27;89"  encodes intervals
    [14, 27] and [45, 89].
    """
    starts = [int(s) for s in str(epitope_start).split(";")]
    ends   = [int(e) for e in str(epitope_end).split(";")]
    label  = np.zeros(len(sequence), dtype=np.int32)
    for s, e in zip(starts, ends):
        label[s - 1 : e] = 1  # 1-indexed inclusive → 0-indexed Python slice
    return label


records = []
degenerate_rows = []

for _, row in raw_df.iterrows():
    label_vec  = parse_epitope_label(row["sequence"], row["epitope_start"], row["epitope_end"])
    n_epitope  = int(label_vec.sum())
    seq_len    = len(row["sequence"])
    
    if n_epitope == 0:
        degenerate_rows.append({"accession": row["accession"], "reason": "no_epitope_residues", "seq_len": seq_len, "n_epitope": n_epitope})
        continue
    if n_epitope == seq_len:
        degenerate_rows.append({"accession": row["accession"], "reason": "all_epitope_residues", "seq_len": seq_len, "n_epitope": n_epitope})
        continue

    records.append({
        "accession":          row["accession"],
        "sequence":           row["sequence"],
        "epitope_label":      label_vec,
        "seq_len":            seq_len,
        "n_epitope_residues": n_epitope,
        "epitope_density":    n_epitope / seq_len,
    })

if degenerate_rows:
    deg_df = pd.DataFrame(degenerate_rows)
    print(f"Filtered {len(deg_df)} degenerate proteins (uninformative for residue-level ranking):")
    print(deg_df.to_string(index=False))
else:
    print("No degenerate proteins found.")

eval_df = pd.DataFrame(records).reset_index(drop=True)
print(f"Proteins after filtering degenerate cases: {len(eval_df)}")

No degenerate proteins found.
Proteins after filtering degenerate cases: 123


## Model — FrozenESMAllergenClassifier

Class definitions copied verbatim from notebook 03.

In [10]:
tokenizer = build_tokenizer(HF_MODEL_NAME)


In [11]:
model, checkpoint = load_baseline_checkpoint(
    CHECKPOINT_PATH,
    DEVICE,
    model_name=HF_MODEL_NAME,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
)
arch_hp = checkpoint.get(
    "architecture_hyperparameters",
    {"hidden_dim": HIDDEN_DIM, "dropout": DROPOUT},
)

print(f"Loaded checkpoint:  {CHECKPOINT_PATH}")
print(f"ESM model:          {checkpoint.get('esm_model_name', ESM_MODEL_NAME)}")
print(f"hidden_dim={arch_hp.get('hidden_dim', HIDDEN_DIM)}, dropout={arch_hp.get('dropout', DROPOUT)}")


Some weights of EsmModel were not initialized from the model checkpoint at /Users/jianzhouyao/.cache/huggingface/hub/models--facebook--esm2_t6_8M_UR50D/snapshots/c731040fcd8d73dceaa04b0a8e6329b345b0f5df and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded checkpoint:  /Users/jianzhouyao/Library/Mobile Documents/com~apple~CloudDocs/Universität/ETH/DL in Biology/XAllergen2.0/models/baseline_frozen_esm2.pt
ESM model:          esm2_t6_8M_UR50D
hidden_dim=128, dropout=0.3


## Attribution Methods

In [12]:
def _precision_at_k(y_true: np.ndarray, scores: np.ndarray) -> float:
    k = int(y_true.sum())
    if k == 0:
        return float("nan")
    top_k = np.argsort(scores)[::-1][:k]
    return float(y_true[top_k].sum() / k)


def compute_metrics(y_true: np.ndarray, scores: np.ndarray) -> dict:
    if len(np.unique(y_true)) < 2:
        auroc = float("nan")
    else:
        auroc = float(roc_auc_score(y_true, scores))
    return {
        "auroc": auroc,
        "auprc": float(average_precision_score(y_true, scores)),
        "precision_at_k": _precision_at_k(y_true, scores),
    }


## Evaluation Loop

Processes proteins one at a time (memory-safe for IG). Skips sequences longer than 1022 tokens.

In [13]:
print(eval_df.shape)
print(eval_df.columns.tolist())
print(eval_df.head())

(123, 6)
['accession', 'sequence', 'epitope_label', 'seq_len', 'n_epitope_residues', 'epitope_density']
  accession                                           sequence  \
0    Q84ZX5  MAKCSYVFCAVLLIFIVAIGEMEAAGSKLCEKTSKTYSGKCDNKKC...   
1    P49372  MGVQTHVLELTSSVSAEKIFQGFVIDVDTVLPKAAPGAYKSVEIKG...   
2    P67875  MVAIKNLFLLAATAVSVLAAPSPLDARATWTCINQQLNPKTNKWED...   
3    P01012  MGSIGAASMEFCFDVFKELKVHHANENIFYCPIAIMSALAMVYLGA...   
4    P80384  MMKFIALFALVAVASAGKMTFKDCGHGEVTELDITGCSGDTCVIHR...   

                                       epitope_label  seq_len  \
0  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...      132   
1  [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...      154   
2  [0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...      176   
3  [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...      386   
4  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...      141   

   n_epitope_residues  epitope_density  
0                  99         0.750000  
1                  99     

In [14]:
rng = np.random.default_rng(RANDOM_STATE)

results_rows = []
n_skipped = 0

for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Evaluating proteins"):
    sequence = row["sequence"]
    epitope_labels = row["epitope_label"]
    accession = row["accession"]
    seq_len = row["seq_len"]

    tok_len = tokenizer(sequence, add_special_tokens=False, return_tensors="pt")["input_ids"].shape[1]
    if tok_len > MAX_SEQ_LEN:
        n_skipped += 1
        continue

    base = {
        "accession": accession,
        "seq_len": seq_len,
        "epitope_density": row["epitope_density"],
        "n_epitope_residues": row["n_epitope_residues"],
    }

    attn_scores = None
    try:
        attn_scores = compute_attention_weights(model, tokenizer, sequence, DEVICE)
        results_rows.append(
            {**base, "method": "attention_weights", **compute_metrics(epitope_labels, attn_scores)}
        )
    except Exception as exc:
        print(f"[attention] {accession}: {exc}")

    try:
        ig_scores = compute_integrated_gradients(
            model,
            tokenizer,
            sequence,
            DEVICE,
            steps=IG_STEPS,
            normalize=False,
        )
        results_rows.append(
            {**base, "method": "integrated_gradients", **compute_metrics(epitope_labels, ig_scores)}
        )
    except Exception as exc:
        print(f"[IG] {accession}: {exc}")

    rand_metrics = [
        compute_metrics(epitope_labels, rng.uniform(0.0, 1.0, size=seq_len))
        for _ in range(N_RANDOM_DRAWS)
    ]
    results_rows.append({
        **base,
        "method": "random_mean",
        **mean_metric_dicts(rand_metrics),
    })

    if attn_scores is not None:
        try:
            shuffled_metrics = [
                compute_metrics(rng.permutation(epitope_labels), attn_scores)
                for _ in range(N_RANDOM_DRAWS)
            ]
            results_rows.append({
                **base,
                "method": "shuffled_mean",
                **mean_metric_dicts(shuffled_metrics),
            })
        except Exception as exc:
            print(f"[shuffled] {accession}: {exc}")

print(f"\nSkipped {n_skipped} proteins exceeding MAX_SEQ_LEN={MAX_SEQ_LEN}")
results_df = pd.DataFrame(results_rows)
print(f"Total result rows: {len(results_df)}")
print(results_df["method"].value_counts().to_string())


Evaluating proteins:   1%|          | 1/123 [00:00<01:18,  1.55it/s]

[IG] Q84ZX5: 'bool' object has no attribute 'unsqueeze'
[IG] P49372: 'bool' object has no attribute 'unsqueeze'


Evaluating proteins:   2%|▏         | 3/123 [00:01<00:35,  3.37it/s]

[IG] P67875: 'bool' object has no attribute 'unsqueeze'


Evaluating proteins:   3%|▎         | 4/123 [00:01<00:39,  2.98it/s]

...

[IG] K6ZYR9: 'bool' object has no attribute 'unsqueeze'
[IG] K7AF06: 'bool' object has no attribute 'unsqueeze'


Evaluating proteins:  50%|█████     | 62/123 [00:13<00:11,  5.21it/s]

[IG] K7AJI8: 'bool' object has no attribute 'unsqueeze'
[IG] K7AEU6: 'bool' object has no attribute 'unsqueeze'


Evaluating proteins:  50%|█████     | 62/123 [00:13<00:13,  4.56it/s]


KeyboardInterrupt: 

## Figures

In [ ]:
PALETTE = {
    "attention_weights":    "#4C72B0",
    "integrated_gradients": "#DD8452",
    "random_mean":          "#55A868",
}
METHOD_XLABELS = {
    "attention_weights":    "Attention\nWeights",
    "integrated_gradients": "Integrated\nGradients",
    "random_mean":          "Random\nMean",
}

violin_order   = ["attention_weights", "integrated_gradients", "random_mean"]
violin_df      = results_df[results_df["method"].isin(violin_order)].copy()
metrics_config = [
    ("auroc",          "AUROC"),
    ("auprc",          "AUPRC"),
    ("precision_at_k", "Precision@k"),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

for ax, (col, label) in zip(axes, metrics_config):
    plot_data = violin_df.dropna(subset=[col]).copy()

    sns.violinplot(
        data=plot_data, x="method", y=col,
        palette=PALETTE, inner=None, cut=0, order=violin_order, ax=ax,
    )
    sns.stripplot(
        data=plot_data, x="method", y=col,
        palette=PALETTE, alpha=0.3, size=4, jitter=True, order=violin_order, ax=ax,
    )

    rand_mean = plot_data.loc[plot_data["method"] == "random_mean", col].mean()
    ax.axhline(
        rand_mean, color="gray", linestyle="--", linewidth=1.2,
        label=f"Random mean = {rand_mean:.3f}",
    )

    overall_mean = plot_data[col].mean()
    overall_sd   = plot_data[col].std()
    ax.set_title(f"{label}\nmean\u00b1SD: {overall_mean:.3f} \u00b1 {overall_sd:.3f}", fontsize=11)
    ax.set_xlabel("Method")
    ax.set_ylabel(label)
    ax.set_xticklabels([METHOD_XLABELS[m] for m in violin_order], fontsize=9)
    ax.legend(fontsize=8)

plt.suptitle("Residue Attribution Faithfulness vs. IEDB Epitopes", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(VIOLINS_PNG, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {VIOLINS_PNG}")

In [ ]:
scatter_methods = ["attention_weights", "integrated_gradients", "random_mean"]
scatter_df      = results_df[results_df["method"].isin(scatter_methods)].copy()

for metric_col, metric_label, out_path in [
    ("auroc", "AUROC", AUROC_DENSITY_PNG),
    ("auprc", "AUPRC", AUPRC_DENSITY_PNG),
]:
    fig, ax = plt.subplots(figsize=(9, 6))

    for method in scatter_methods:
        mdf   = scatter_df[scatter_df["method"] == method].dropna(
            subset=[metric_col, "epitope_density"]
        )
        color = PALETTE[method]
        ax.scatter(
            mdf["epitope_density"], mdf[metric_col],
            color=color, alpha=0.4, s=30,
            label=method.replace("_", " ").title(),
        )
        if len(mdf) >= 5:
            smoothed = lowess(
                mdf[metric_col].values, mdf["epitope_density"].values,
                frac=0.5, return_sorted=True,
            )
            ax.plot(smoothed[:, 0], smoothed[:, 1], color=color, linewidth=2.0)

    ax.set_xlabel("Epitope Density (fraction of residues)", fontsize=12)
    ax.set_ylabel(metric_label, fontsize=12)
    ax.set_title(f"{metric_label} vs. Epitope Density", fontsize=13)
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path}")

## Summary Table

In [ ]:
summary_rows = []
for method in ["attention_weights", "integrated_gradients", "random_mean", "shuffled_mean"]:
    mdf = results_df[results_df["method"] == method]
    entry = {"method": method, "n_proteins": len(mdf)}
    for col in ["auroc", "auprc", "precision_at_k"]:
        vals = mdf[col].dropna()
        entry[f"{col}_mean"] = round(float(vals.mean()), 4)
        entry[f"{col}_sd"]   = round(float(vals.std()),  4)
    summary_rows.append(entry)

summary_df = pd.DataFrame(summary_rows)[[
    "method",
    "auroc_mean", "auroc_sd",
    "auprc_mean", "auprc_sd",
    "precision_at_k_mean", "precision_at_k_sd",
    "n_proteins",
]]

print(summary_df.to_string(index=False))
summary_df.to_csv(SUMMARY_CSV, index=False)
print(f"\nSaved: {SUMMARY_CSV}")

In [ ]:
if RUN_TARGET == "colab":
    import shutil as _shutil
    DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
    for _src in [SUMMARY_CSV, VIOLINS_PNG, AUROC_DENSITY_PNG, AUPRC_DENSITY_PNG]:
        if _src.exists():
            _shutil.copy2(_src, DRIVE_RESULTS / _src.name)
            print(f"Copied to Drive: {DRIVE_RESULTS / _src.name}")
        else:
            print(f"WARNING: {_src.name} not found \u2014 run all cells first.")
    print()
    print("Results in Google Drive > My Drive > XAllergen2.0 > results/")
    print("Sync via Google Drive for Desktop or download from drive.google.com.")
else:
    print("Local run: results saved to:")
    for _out_path in [SUMMARY_CSV, VIOLINS_PNG, AUROC_DENSITY_PNG, AUPRC_DENSITY_PNG]:
        print(f"  {_out_path}")
